# MedTrack_DV – Hospital Operations & Patient Analytics
## Milestone 1: Data Cleaning & Transformation Notebook

This notebook executes a complete 25-step data cleaning pipeline to convert raw operational hospital records (`hospital_raw_data.csv`) into a high-quality, Tableau-ready analytical dataset (`hospital_cleaned.csv`).

### 1. Import Libraries

In [9]:
!pip install numpy
!pip install pandas

import numpy as np
import pandas as pd


ImportError: DLL load failed while importing missing: An Application Control policy has blocked this file.

In [8]:
import pandas as pd
import numpy as np



ImportError: DLL load failed while importing missing: An Application Control policy has blocked this file.

### 2. Load Dataset

In [ ]:
raw_path = '../data/raw/hospital_raw_data.csv' if os.path.exists('../data/raw/hospital_raw_data.csv') else 'data/raw/hospital_raw_data.csv'
df = pd.read_csv(raw_path, dtype=str)
print(f'Successfully loaded raw dataset from {raw_path}')

### 3. Dataset Overview

In [ ]:
df.head(5)

### 4. Shape

In [ ]:
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')

### 5. Column Names

In [ ]:
print(list(df.columns))

### 6. Data Types

In [ ]:
df.dtypes

### 7. Descriptive Statistics

In [ ]:
df.describe(include='all')

### 8. Missing Value Analysis

In [ ]:
missing = df.isna().sum() + (df == '').sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct': missing_pct})
missing_df[missing_df['Missing_Count'] > 0]

### 9. Duplicate Detection

In [ ]:
dup_ids = df.duplicated(subset=['Admission_ID'], keep=False)
print(f'Total duplicate records based on Admission_ID: {dup_ids.sum()}')

### 10. Duplicate Removal

In [ ]:
initial_count = len(df)
df = df.drop_duplicates(subset=['Admission_ID'], keep='first').reset_index(drop=True)
print(f'Removed {initial_count - len(df)} duplicate records. New record count: {len(df)}')

### 11. Missing Value Treatment

In [ ]:
df['Treatment'] = df['Treatment'].replace('', np.nan).fillna('Standard Protocol')
df['Doctor_ID'] = df['Doctor_ID'].replace('', np.nan).fillna('DOC-999')
df['Room_ID'] = df['Room_ID'].replace('', np.nan).fillna('RM-100')

### 12. Department Name Standardization

In [ ]:
dept_clean_map = {
    'general medicine': 'General Medicine', 'gen med': 'General Medicine', 'general medicine ': 'General Medicine',
    'cardiology': 'Cardiology', 'cardiology ': 'Cardiology',
    'neurology': 'Neurology', 'neurology ': 'Neurology',
    'orthopedics': 'Orthopedics', 'orthopedics ': 'Orthopedics',
    'pediatrics': 'Pediatrics', 'pediatrics ': 'Pediatrics',
    'emergency': 'Emergency', 'emergency ': 'Emergency', 'er': 'Emergency',
    'icu': 'ICU', 'icu ': 'ICU', 'intensive care': 'ICU',
    'surgery': 'Surgery', 'surgery ': 'Surgery'
}
df['Department'] = df['Department'].str.strip().apply(
    lambda x: dept_clean_map.get(str(x).lower(), str(x).strip().title() if pd.notna(x) else 'General Medicine')
)
print('Standardized Departments:', sorted(df['Department'].unique()))

### 13. Hospital Name Standardization

In [ ]:
df['Hospital_Name'] = df['Hospital_Name'].str.strip()
print('Unique Standardized Hospitals:', df['Hospital_Name'].nunique())

### 14. Gender Standardization

In [ ]:
gender_map = {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female', 'other': 'Other'}
df['Gender'] = df['Gender'].str.strip().str.lower().map(lambda x: gender_map.get(x, 'Other'))
print('Gender Value Counts:\n', df['Gender'].value_counts())

### 15. Admission Type Standardization

In [ ]:
adm_type_map = {'emergency': 'Emergency', 'urgent': 'Urgent', 'elective': 'Elective'}
df['Admission_Type'] = df['Admission_Type'].str.strip().str.lower().map(lambda x: adm_type_map.get(x, 'Emergency'))
print('Admission Types:', df['Admission_Type'].unique())

### 16. Patient Type Standardization

In [ ]:
pat_type_map = {'inpatient': 'Inpatient', 'outpatient': 'Outpatient', 'emergency': 'Emergency', 'day care': 'Day Care'}
df['Patient_Type'] = df['Patient_Type'].str.strip().str.lower().map(lambda x: pat_type_map.get(x, 'Inpatient'))
print('Patient Types:', df['Patient_Type'].unique())

### 17. Date Conversion

In [ ]:
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'], errors='coerce')
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'], errors='coerce')
df['Readmission_Date'] = pd.to_datetime(df['Readmission_Date'], errors='coerce')

### 18. Invalid Date Detection

In [ ]:
invalid_dates = df[df['Discharge_Date'] < df['Admission_Date']]
print(f'Invalid date pairs (Discharge < Admission): {len(invalid_dates)}')
# Correcting invalid dates
df.loc[df['Discharge_Date'] < df['Admission_Date'], 'Discharge_Date'] = df['Admission_Date'] + pd.Timedelta(days=5)

### 19. Length of Stay Validation

In [ ]:
df['Length_of_Stay'] = (df['Discharge_Date'] - df['Admission_Date']).dt.days
df['Length_of_Stay'] = df['Length_of_Stay'].clip(lower=1)
print('Length of Stay Summary:')
print(df['Length_of_Stay'].describe())

### 20. Bed Capacity Validation

In [ ]:
df['Total_Beds'] = pd.to_numeric(df['Total_Beds'], errors='coerce').astype(int)
df['Occupied_Beds'] = pd.to_numeric(df['Occupied_Beds'], errors='coerce').astype(int)
# Cap Occupied_Beds to Total_Beds if overflow exists
df.loc[df['Occupied_Beds'] > df['Total_Beds'], 'Occupied_Beds'] = df['Total_Beds']
df['Available_Beds'] = df['Total_Beds'] - df['Occupied_Beds']
print('Bed capacity validation completed. Negative available beds count:', (df['Available_Beds'] < 0).sum())

### 21. Resource Data Validation

In [ ]:
for col in ['Staff_Count', 'Doctor_Count', 'Nurse_Count', 'Equipment_Total', 'Equipment_In_Use']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
print('Resource Data Summary:')
df[['Staff_Count', 'Doctor_Count', 'Nurse_Count', 'Equipment_Total', 'Equipment_In_Use']].describe()

### 22. Numerical Range Validation

In [ ]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
median_age = int(df['Age'][(df['Age'] >= 0) & (df['Age'] <= 110)].median())
df.loc[(df['Age'] < 0) | (df['Age'] > 110) | (df['Age'].isna()), 'Age'] = median_age
df['Age'] = df['Age'].astype(int)

df['Treatment_Cost'] = pd.to_numeric(df['Treatment_Cost'], errors='coerce').abs().round(2)
print('Age Range:', df['Age'].min(), 'to', df['Age'].max())
print('Treatment Cost Range: $', df['Treatment_Cost'].min(), 'to $', df['Treatment_Cost'].max())

### 23. Outlier Detection

In [ ]:
q1 = df['Treatment_Cost'].quantile(0.25)
q3 = df['Treatment_Cost'].quantile(0.75)
iqr = q3 - q1
high_cost_cases = df[df['Treatment_Cost'] > (q3 + 1.5 * iqr)]
print(f'Detected {len(high_cost_cases)} complex high-cost cases (retained as valid healthcare ICU/Surgical procedures).')

### 24. Data Type Optimization

In [ ]:
df['Admission_Date'] = df['Admission_Date'].dt.strftime('%Y-%m-%d')
df['Discharge_Date'] = df['Discharge_Date'].dt.strftime('%Y-%m-%d')
df['Readmission_Date'] = df['Readmission_Date'].dt.strftime('%Y-%m-%d').fillna('')
df.info()

### 25. Final Quality Validation

In [ ]:
cleaned_records = len(df)
cleaned_cells = cleaned_records * len(df.columns)
missing_cells = sum(df[col].isin(['', np.nan]).sum() for col in df.columns if col != 'Readmission_Date')
completeness_pct = round(((cleaned_cells - missing_cells) / cleaned_cells) * 100, 2)
missing_pct = round((missing_cells / cleaned_cells) * 100, 2)

print('=' * 60)
print('FINAL CLEANED DATASET METRICS')
print('=' * 60)
print(f'Cleaned Record Count    : {cleaned_records:,}')
print(f'Dataset Completeness    : {completeness_pct}% (Target: > 95%)')
print(f'Missing Value Percentage: {missing_pct}% (Target: < 2%)')
print(f'Duplicates              : {df.duplicated(subset=["Admission_ID"]).sum()}')
print('=' * 60)
if completeness_pct > 95.0 and missing_pct < 2.0:
    print('MILESTONE 1 STATUS: PASSED')
else:
    print('MILESTONE 1 STATUS: FAILED')